# Amuse Web — Google Colab

NVIDIA GPUでLinux Web版を動かすノートブックです。「ランタイム → ランタイムのタイプを変更」でGPUを選び、上から順に実行します。初回は.NETとCUDAライブラリーのビルドに時間がかかります。

このノートブックから `https://github.com/cc-h-kuwabara/AmuseAI-WebUI.git` の `webui` ブランチをcloneしてビルドします。ソースZIPのアップロードは不要です。Pythonのグローバルパッケージは変更しません。

Web UIはCloudflare Quick Tunnelで公開します。「3. 起動・表示」に出るリンクをブラウザーで開いてください。Colabのポートプロキシ・iframeは使いません。公開URLを知る人はアプリを操作できます（ログイン認証なし）。URLを共有しないでください。常時起動・切断回避処理は使いません。[Colab公式FAQ](https://research.google.com/colaboratory/faq.html)では、無料枠・コンピュートユニット残高なしでWeb UIを主に使う生成処理が制限されています。その場合は `USE_WEB_UI=False` とし、下の生成セルから実行してください。有料枠でもランタイムの寿命・利用可能GPUは保証されません。

モデル・結果は通常ランタイム終了時に失われます。必要ならDriveへモデルを保存し、最後のセルで結果をダウンロードします。Colab上のCUDA実推論は別途実機確認が必要です。


In [ ]:
from pathlib import Path
WORK = Path('/content/amuse')
PORT = 8082
USE_GPU = True
USE_WEB_UI = True
MOUNT_DRIVE = False
CACHE_DIRECTORY = ''  # Drive使用時は MyDrive/Amuse/BuildCache を自動使用
BUILD_JOBS = 0  # 0: CPU・空きメモリーに応じて自動設定（最大8）
MODEL_DIRECTORY = ''  # 例: /content/drive/MyDrive/Amuse/Models
GIT_URL = 'https://github.com/cc-h-kuwabara/AmuseAI-WebUI.git'
GIT_REF = 'webui'  # Colab対応を含むブランチ
SOURCE = WORK / 'source'
DATA = WORK / 'data'


## 1. GitHubからcloneする
指定リポジトリーのwebuiブランチを取得します。再実行時は既存のcloneを再利用し、実行するコミットを表示します。Driveのマウントは任意です。


In [ ]:
import sys, subprocess
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if CACHE_DIRECTORY.startswith('/content/drive/') and not MOUNT_DRIVE:
    raise ValueError('Driveキャッシュには MOUNT_DRIVE=True が必要です。')
if MODEL_DIRECTORY.startswith('/content/drive/') and not MOUNT_DRIVE:
    raise ValueError('Driveのモデル保存先を使う場合は MOUNT_DRIVE=True にしてください。')
WORK.mkdir(parents=True, exist_ok=True)
if SOURCE.exists():
    if not (SOURCE / '.git').is_dir():
        raise RuntimeError('保存先がGit cloneではありません。WORKを別のフォルダーに変更してください。')
    remote = subprocess.check_output(['git', '-C', str(SOURCE), 'remote', 'get-url', 'origin'], text=True).strip()
    branch = subprocess.check_output(['git', '-C', str(SOURCE), 'branch', '--show-current'], text=True).strip()
    if remote != GIT_URL or branch != GIT_REF:
        raise RuntimeError('保存先のリポジトリーまたはブランチが設定と異なります。WORKを別のフォルダーに変更してください。')
    print('既存のcloneを再利用:', SOURCE)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, GIT_URL, str(SOURCE)], check=True)
if not (SOURCE / 'Scripts/colab_runtime.py').is_file():
    raise RuntimeError('このブランチにはColab対応がありません。GIT_REFを確認してください。')
print('実行コミット:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())
sys.path.insert(0, str(SOURCE / 'Scripts'))
import colab_runtime as amuse
print('Source ready:', SOURCE)


## 2. ビルドする
依存ライブラリー、プロジェクト専用.NET SDK、Web版、AIホストを準備します。GPUを使う場合は固定版StableDiffusion.cppのCUDA共有ライブラリーも構築します。モデルのダウンロードはまだ行いません。


高速化するには最初の設定セルで `MOUNT_DRIVE=True` にしてください。ビルド済みアプリとCUDAライブラリーをDriveに保存し、次回は互換性のあるキャッシュから復元します。GPU・CUDA・OSやソースが変わると必要な部分を再ビルドします。モデルや認証情報はビルドキャッシュに含めません。
初回のCUDAコンパイルは必要です。`BUILD_JOBS=0` は並列数を自動設定します。メモリー不足で失敗する場合は `BUILD_JOBS=2` に下げてください。同じランタイム内の再実行では既存のビルドを利用します。


In [ ]:
print('ビルドログ:', WORK / 'logs/build.log')
cache_directory = CACHE_DIRECTORY or ('/content/drive/MyDrive/Amuse/BuildCache' if MOUNT_DRIVE else None)
distribution, DATA = amuse.prepare(SOURCE, WORK, use_gpu=USE_GPU, cache_directory=cache_directory, build_jobs=BUILD_JOBS)
print('配布版:', distribution)
print('データ:', DATA)


## 3. 起動・表示
ポートは設定セルのPORTで変更できます。同じセルの再実行時は、このノートブックで起動したサーバーを停止してから再起動します。別のプロセスは停止しません。


In [ ]:
if 'amuse_server' in globals():
    amuse_server.stop()
if 'amuse_tunnel' in globals():
    amuse_tunnel.stop()
proxy_url = None
try:
    if USE_WEB_UI:
        amuse_tunnel = amuse.CloudflareTunnel(WORK, PORT).start()
        proxy_url = amuse_tunnel.url
    amuse_server = amuse.Server(distribution, DATA, PORT, proxy_url=proxy_url).start()
except BaseException:
    if 'amuse_tunnel' in globals():
        amuse_tunnel.stop()
    raise
amuse_server.initialize_environment(use_gpu=USE_GPU)
if MODEL_DIRECTORY:
    general = amuse_server.request('/api/config/general')
    general['directoryModel'] = MODEL_DIRECTORY
    amuse_server.wait_job(amuse_server.request('/api/config/general', general, method='PUT'))
catalog = amuse_server.request('/api/catalog')
print('実行デバイス:', [(d['id'], d['name']) for d in catalog['devices']])
if USE_WEB_UI:
    from IPython.display import display, Markdown
    display(Markdown(f'[Open Amuse Web UI]({proxy_url})'))
else:
    print('ノートブックAPIモードです。次の生成セルを使用してください。')


## 4. セルから画像を生成する（任意）

`RUN_SAMPLE=True` にすると、FLUX.2 Klein 4B（ID 550）と必要コンポーネントを導入し、256×256・4ステップで画像を生成します。初回ダウンロードは数GBになるため、ディスクと利用時間を確保してください。Web UIから同じモデルを導入済みなら再ダウンロードはしません。

モデルにトークンが必要な場合はWeb UIの設定から入力してください。トークンをソースや出力セルに記録しないでください。


In [ ]:
RUN_SAMPLE = False
PROMPT = 'A small blue ceramic teapot on a wooden desk, soft daylight, original illustration'
if RUN_SAMPLE:
    from IPython.display import display, Image
    result = amuse.sample_image(amuse_server, PROMPT, use_gpu=USE_GPU)
    media = {f['id']: f for f in amuse_server.request('/api/files')}
    for ident in result['files']:
        if media[ident]['contentType'].startswith('image/'):
            display(Image(data=amuse_server.request('/api/files/' + ident, raw=True)))


## 5. 状態確認・結果の保存（任意）
結果ZIPには画像・入力ファイル・生成レシピが含まれます。認証トークンを含むSettings.jsonは収録しません。


In [ ]:
print([(j['id'], j['status'], j['message']) for j in amuse_server.request('/api/jobs')[:10]])
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files
    archive = shutil.make_archive(str(WORK / 'Amuse_Results'), 'zip', DATA / 'Files')
    files.download(archive)
# 障害調査時だけ実行: print(amuse_server.logs())


## 6. 停止
処理が終わったらこのセルで停止し、不要ならColabランタイムも切断してください。


In [ ]:
if 'amuse_server' in globals():
    amuse_server.stop()
    print('Amuseを停止しました。')
if 'amuse_tunnel' in globals():
    amuse_tunnel.stop()
